# GPU Programming: A Mental Model for Understanding PyTorch

**Goal:** Understand what happens beneath a PyTorch operation such as `y = x @ W`, not become a GPU performance engineer.

## 1. Execution hierarchy

```text
Grid (one kernel launch)
  └── Thread Blocks (CUDA) / Threadgroups (Metal)
        └── Warps (CUDA) / SIMD-groups (Metal)
              └── Threads
```

- **Kernel:** A function whose code runs on the GPU. **Kernel launch / dispatch:** One invocation of that function across many threads.
- **Grid:** All threads launched for one kernel invocation, arranged into blocks.
- **Thread block:** A group of threads that can cooperate through **shared memory** and block-level synchronization.
- **Warp:** On NVIDIA GPUs, typically **32 threads** that execute instructions as a group (SIMT). All threads run the same kernel code, but their IDs and data differ. Branches can cause **warp divergence**, so different paths may execute with some lanes masked.
- **Thread:** A logical execution instance with its own IDs and per-thread state.

**SM (Streaming Multiprocessor)** is *hardware*, not another level in that logical hierarchy. A GPU has multiple SMs. The GPU schedules whole blocks onto SMs; one SM can host multiple blocks at once, subject to available resources. Blocks of the same grid may run on different SMs, and blocks need not run simultaneously or in a particular order.

## 2. Memory hierarchy

| Memory | Where / what it is | Scope and purpose |
|---|---|---|
| **Global / device memory** | On a discrete NVIDIA GPU, primarily its **VRAM** (HBM or GDDR). On Apple silicon, GPU-accessible **unified memory** uses the same physical DRAM pool as the CPU. | Large storage for tensor data, model weights, activations, and outputs; buffers can survive across kernel launches. |
| **Shared memory** (CUDA) / **threadgroup memory** (Metal) | Small, fast on-chip memory associated with an SM / GPU compute unit | A private region for each resident block/threadgroup, shared among its threads; useful for cooperation and **tiling**. |
| **Registers** | Very fast on-chip storage | Per-thread working values; allocation is largely handled by the compiler and hardware. |
| **L1 / L2 caches** | On-chip caches | Usually hardware-managed; reduce repeated access to device memory. |

**Important:** Apple *unified memory* is **not** Metal *threadgroup memory*. Unified memory is the large backing memory; threadgroup memory is small on-chip scratch space.

### Who allocates memory?

- The **CPU application** asks a runtime / driver to allocate device buffers (e.g., `cudaMalloc`); the allocation can be used by multiple kernels. A kernel generally writes to **preallocated output memory**.
- The **programmer can request shared memory per block**, either statically (`__shared__ int tile[4];`) or dynamically (`extern __shared__ int tile[];` and a byte count in the kernel launch).
- The GPU allocates the requested *on-chip resources when scheduling a block*. Shared memory is **not** an ordinary chunk of VRAM, and it is released for reuse when the block finishes.
- Registers are generally chosen by the compiler; caches are largely automatic.
- The **host OS + GPU driver**, together with GPU firmware/hardware, manage resources and scheduling. A GPU normally does not run a separate user-installable desktop OS. A CUDA context is the GPU execution/resource environment, not a new OS process per kernel.

### Occupancy (know the idea, not the tuning details)

**Occupancy** is the fraction of an SM's maximum supported resident warps that are currently resident. More registers or shared memory **per block** can reduce the number of blocks/warps that fit on an SM. Higher occupancy can help hide latency, but **maximum occupancy does not necessarily mean maximum performance**.

## 3. End-to-end CUDA example: one small grid, two blocks

**Task:** Read eight integers on the CPU, move them to GPU memory, and have each thread add its input to the *next element in its own block*. This deliberately uses shared memory to show how threads cooperate.

```cpp
#include <cstdio>
#include <cuda_runtime.h>

__global__ void neighbor_sum(const int* input, int* output) {
    extern __shared__ int tile[];    // One separate tile per block

    int t = threadIdx.x;             // Position within this block
    int i = blockIdx.x * blockDim.x + t; // Position in the grid

    tile[t] = input[i];              // Global memory -> shared memory
    __syncthreads();                 // Wait for all block writes

    output[i] = tile[t] + tile[(t + 1) % blockDim.x];
                                    // Shared memory -> global output
}

int main() {
    int h_input[8]  = {1, 2, 3, 4, 5, 6, 7, 8};
    int h_output[8] = {};
    int *d_input, *d_output;

    cudaMalloc((void**)&d_input, sizeof(h_input));
    cudaMalloc((void**)&d_output, sizeof(h_output));
    cudaMemcpy(d_input, h_input, sizeof(h_input),
               cudaMemcpyHostToDevice);

    neighbor_sum<<<2, 4, 4 * sizeof(int)>>>(d_input, d_output);

    cudaMemcpy(h_output, d_output, sizeof(h_output),
               cudaMemcpyDeviceToHost);
    for (int value : h_output) std::printf("%d ", value);
    std::printf("\n");

    cudaFree(d_input);
    cudaFree(d_output);
}
```

Compile and run on a system with a CUDA-capable NVIDIA GPU and CUDA Toolkit:

```bash
nvcc demo.cu -o demo
./demo
```

Output:

```text
3 5 7 5 11 13 15 13
```

### Trace the full lifecycle

1. **CPU owns the host arrays:** `h_input` and `h_output` reside in CPU-accessible memory.
2. **CPU requests GPU buffers:** `cudaMalloc` obtains device-memory allocations for `d_input` and `d_output` (typically VRAM on a discrete NVIDIA GPU).
3. **CPU transfers input:** `cudaMemcpy(...HostToDevice)` copies eight integers into `d_input`.
4. **CPU submits work:** `neighbor_sum<<<2, 4, 16>>>` launches a grid of **2 blocks × 4 threads = 8 threads**. Each block requests **16 bytes of dynamic shared memory**. The same kernel code runs for all eight threads.
5. **GPU schedules blocks:** The GPU chooses which SM runs each block, when they run, and how warps are issued. Our code chooses the work per thread using `threadIdx.x`, `blockIdx.x`, and `blockDim.x`; it does **not** select an SM or schedule warps. This toy example uses only four threads per block—intentionally far below a full 32-thread NVIDIA warp.
6. **Threads cooperate:** Each thread copies one number into its block's own `tile`; `__syncthreads()` ensures those writes finish before any thread reads a neighbor's value. The blocks do **not** share the same tile.
7. **GPU writes output:** `output[i] = ...` stores results in **`d_output` in global/device memory**, not directly in CPU RAM.
8. **CPU reads results:** The synchronous Device-to-Host `cudaMemcpy` waits for the preceding work in the default stream as necessary, then copies eight results into `h_output`. `cudaFree` releases the two GPU buffers.
9. A thread block does not need to contain a multiple of 32 threads. On NVIDIA GPUs, a **partially filled warp has inactive lanes**. Warps never span across thread blocks, so small blocks may underutilize SIMD execution resources.
    
*For teaching clarity, this minimal example omits CUDA error checking and boundary checks; its grid is intentionally sized to match the eight inputs exactly.*

### Who controls what?

| Programmer / CPU application chooses | GPU / compiler / driver handles |
|---|---|
| Kernel code and per-thread logic | Mapping blocks to SMs and scheduling warps |
| Grid size and threads per block | When individual blocks execute |
| Device-buffer allocation requests | Physical allocation / address mappings |
| Requested shared memory per block | Assigning on-chip resources to resident blocks |
| When to launch, synchronize, and copy results | Register assignment and much of cache behavior |

## 4. Connect this to PyTorch

```python
import torch

x = torch.tensor([1, 2, 3], device="cuda")
y = x * 2
print(y.cpu())
```

- The **Python Tensor object** tracks metadata such as shape, dtype, device, and strides; its GPU storage contains the numeric data.
- PyTorch's dispatcher/backend chooses GPU implementations and launches kernels (or, when compiled/fused, may combine operations).
- The output tensor `y` generally stays in GPU memory. **`y.cpu()` requests a CPU-side copy**; most intermediate GPU results do **not** need to return to the CPU.
- PyTorch usually uses a **caching allocator** to reuse GPU memory allocations rather than calling the driver for every tensor.

**Bottom line:** **CPU code manages and launches work → GPU blocks run on SMs → threads use registers/shared memory → outputs remain in global/device memory → CPU reads only when needed.**

**Not necessary for this learning goal:** shared-memory bank conflicts, warp shuffle instructions, exact per-SM resource counts, detailed occupancy tuning, or driver/firmware internals.
